In [1]:
import copy

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import wandb


In [2]:
# ==================== 本轮改动说明 ====================
# baseline 的现象：训练 loss 一路降到 0.12，但测试准确率从第 6 轮起就卡在 90.5% 不再上升，
# 甚至连着两轮反着跳。这是典型的“过拟合”——模型把 6 万张训练图背下来了，
# 却没有学到能推广到新图片的规律。baseline 的测试准确率：90.77%。
#
# 本轮改动（按数据 -> 模型 -> 训练的顺序）：
#   1) 训练集加数据增强：随机平移 + 随机水平翻转
#   2) 归一化改用 FashionMNIST 的真实均值/标准差（原来写的 0.5 是拍脑袋定的）
#   3) 从训练集划出 10% 作验证集，用它挑“最好的那一轮”；测试集只在最后评一次
#   4) 模型加深：两组 3x3 卷积堆叠 + BN，全连接前加 Dropout
#   5) 优化器加权重衰减，损失函数加 label smoothing（标签平滑）
#   6) 学习率改成余弦退火，训练轮数 10 -> 35
#
# 踩过的坑（记录一下，避免重复）：
#   一开始只给原来的小网络（16/32 通道）加增强和较重的 Dropout（0.25/0.5），
#   结果模型“学不动”了：训练 loss 卡在 0.51 下不去，测试集只有 86.14%，比 baseline 还低。
#   这属于“正则化过强 + 网络容量不足”造成的欠拟合——正则化是用来压制过拟合的，
#   但压过头就会把模型压到学不会。后来把网络加深、Dropout 调轻才走通。
#
# 本版结果：验证集 94.15%，测试集 93.37%（baseline 90.77%）
# =====================================================

config = {
    "learning_rate": 0.001,
    "batch_size": 64,
    "epochs": 35,               # 改动：10 -> 25 -> 35（配合学习率衰减，训练久一点才收得住）
    "optimizer": "Adam",
    "weight_decay": 5e-5,       # 新增：权重衰减，限制权重变大，属于正则化
    "label_smoothing": 0.1,     # 新增：标签平滑
    "val_ratio": 0.1,           # 新增：划出 10% 训练数据当验证集
    "seed": 42,                 # 新增：固定随机种子，让结果可复现
    "model": "CNN_3x3_BN_FashionMNIST",
}

wandb.init(
    project="fashion-mnist-cnn",   # 项目名，可以自己改
    name="deep-aug-cosine",        # 改动：换名字，和 baseline 那两次实验区分开
    config=config                  # 记录所有超参数
)

# 改动：固定随机种子（权重初始化、数据打乱顺序等都用同一套随机数）
torch.manual_seed(config["seed"])

# ---------- 1. 准备数据 ----------
# 改动：训练集和测试集改用两套不同的预处理流程
# 训练集加数据增强：
#   RandomCrop(28, padding=4)：先在图片四周补 4 圈黑边（28x28 变成 36x36），再随机裁回 28x28，
#                              效果相当于让图片随机上下左右平移，模型就没法死记“第几个像素是什么”
#   RandomHorizontalFlip()：随机左右翻转。衣服左右翻过来还是同一类，所以这样变不会改变标签含义
train_transform = transforms.Compose([
    transforms.RandomCrop(28, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),   # 改动：0.5 -> 数据集真实的均值/标准差
])

# 验证集和测试集只做标准化，绝不做增强，否则评估结果会忽高忽低、不可信
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

# 改动：同一份训练数据加载两份，一份套“带增强”的 transform，一份套“干净”的 transform
train_full = torchvision.datasets.FashionMNIST(root='./data', train=True, transform=train_transform, download=True)
val_full = torchvision.datasets.FashionMNIST(root='./data', train=True, transform=test_transform, download=True)
test_set = torchvision.datasets.FashionMNIST(root='./data', train=False, transform=test_transform, download=True)

# 改动：先切分索引，再把索引分别装进两份数据集，这样验证集不会被增强干扰
generator = torch.Generator().manual_seed(config["seed"])
perm = torch.randperm(len(train_full), generator=generator).tolist()
val_size = int(len(train_full) * config["val_ratio"])
train_set = Subset(train_full, perm[val_size:])
val_set = Subset(val_full, perm[:val_size])

# DataLoader：按批次(Batch)喂数据，每批64张。训练集打乱顺序，验证/测试集不需要打乱
train_loader = DataLoader(train_set, batch_size=config["batch_size"], shuffle=True)
val_loader = DataLoader(val_set, batch_size=config["batch_size"], shuffle=False)
test_loader = DataLoader(test_set, batch_size=config["batch_size"], shuffle=False)

print(f"训练集 {len(train_set)} 张，验证集 {len(val_set)} 张，测试集 {len(test_set)} 张")


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\35314\_netrc.
wandb: Currently logged in as: 3531427435 (3531427435-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run 1x3660el
wandb: Tracking run with wandb version 0.30.0
wandb: Run data is saved locally in E:\code\PycharmProjects\pythonProject\pytorch\wandb\run-20260912_135920-1x3660el
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run deep-aug-cosine
wandb:  View project at https://wandb.ai/3531427435-none/fashion-mnist-cnn
wandb:  View run at https://wandb.ai/3531427435-none/fashion-mnist-cnn/runs/1x3660el


训练集 54000 张，验证集 6000 张，测试集 10000 张


In [3]:
# ---------- 2. 定义模型 ----------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用设备:", device)

class Net(nn.Module):
    # 改动：把原来的两层 5x5 卷积（16/32 通道）换成两组 3x3 卷积堆叠（32/64 通道）。
    # 为什么换：上面记录的那次失败说明原网络“容量不够”，一加正则就学不动。
    # 3x3 卷积堆叠是 VGG 的思路——用两个小卷积核代替一个大卷积核：
    # 感受野差不多，但参数更少、中间多了一次非线性，表达能力反而更强。
    # 28x28 的输入经过两次 2x2 池化变成 7x7，最后接一个小的全连接分类头。
    def __init__(self):
        super().__init__()

        # 第一组卷积：28x28 -> 14x14
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),   # padding=1 让卷积前后尺寸不变
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),   # 改动：卷积特征图上的 Dropout（按通道随机掐），比例很轻
        )

        # 第二组卷积：14x14 -> 7x7
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),   # 改动：同上
        )

        # 分类头：把 64x7x7 的特征摊平后经 128 维隐层输出 10 类
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),     # 改动：全连接层用普通 Dropout，0.5 是这类网络的常见取值
            nn.Linear(128, 10),  # 输出10类
        )

    def forward(self, x):
        x = self.block1(x)      # 卷积组1
        x = self.block2(x)      # 卷积组2
        x = self.classifier(x)  # 分类头
        return x

net = Net().to(device)
print(net)
print("参数量:", sum(p.numel() for p in net.parameters()))
# 改动：log_freq 100 -> 500。log="all" 会定期把每一层的参数、梯度直方图上传到 W&B，
# 频率太高很拖慢训练，这里降低频率，只保留趋势。
wandb.watch(net, log="all", log_freq=500)
# wandb.watch(net, log="gradients", log_freq=100)


使用设备: cuda
Net(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout2d(p=0.1, inplace=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, 

In [4]:
# ---------- 3. 定义损失函数和优化器 ----------
# 改动：交叉熵加 label_smoothing（标签平滑）
# 普通交叉熵要求模型对正确类别给出 100% 的把握，容易过拟合；
# label_smoothing=0.1 会把目标从“1”放宽成“0.9 加上一点点分给其它类”，
# 逼模型不要过度自信，是常见的小幅提分手段。
criterion = nn.CrossEntropyLoss(label_smoothing=config["label_smoothing"])

# optimizer = optim.SGD(net.parameters(), lr=0.04) # 随机梯度下降，学习率0.04
# 改动：加上 weight_decay（权重衰减），限制权重数值不断变大，和 Dropout 一样是抑制过拟合的手段
optimizer = optim.Adam(
    net.parameters(),
    lr=config["learning_rate"],
    weight_decay=config["weight_decay"],
)

# 改动：新增学习率调度器
# 学习率决定“每步把权重调整多少”。一直用固定的学习率，后期会在最优点附近来回跳，
# CosineAnnealingLR 让学习率按余弦曲线从初值平滑降到接近 0：前期步子大、后期步子小，收敛更稳。
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"])


In [5]:
def evaluate(model, loader):
    '''返回模型在给定数据集上的准确率(%)。
    评估时必须切到 eval() 模式：BatchNorm 用固定的统计量，Dropout 关闭。'''
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

# ---------- 4. 训练循环（铁打的5步舞曲） ----------
num_epochs = config["epochs"]
best_val_acc = 0.0    # 改动：记录验证集上最好的准确率
best_state = None     # 改动：把“最好那一轮”的权重存下来

for epoch in range(num_epochs):
    net.train()  # <--- 确保训练时 BatchNorm 正常更新，Dropout 生效
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = net(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    # 改动：看验证集准确率，而不是直接看测试集
    val_acc = evaluate(net, val_loader)

    # 改动：每轮结束后按余弦曲线下调一次学习率
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    # 改动：只保留验证集表现最好的那一份权重，避免“最后一轮反而最差”
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(net.state_dict())

    print(
        f"Epoch [{epoch+1}/{num_epochs}], "
        f"Loss: {avg_loss:.4f}, "
        f"Val Acc: {val_acc:.2f}%, "
        f"Best Val: {best_val_acc:.2f}%, "
        f"LR: {current_lr:.6f}"
    )

    # 记录到 W&B（改动：换成验证集指标，另外记录学习率变化）
    wandb.log({
        "epoch": epoch + 1,
        "train/loss": avg_loss,
        "val/accuracy": val_acc,
        "learning_rate": current_lr,
    })

print("Finished Training")

# ---------- 5. 用验证集挑出来的最优权重评估测试集 ----------
# 改动：测试集只在训练全部结束后评一次，不再参与“挑哪一轮最好”。
# 测试集相当于期末考卷，提前拿它来挑模型等于偷看答案，那样报出来的分数是不可信的。
net.load_state_dict(best_state)
test_acc = evaluate(net, test_loader)
print(f"验证集最佳准确率: {best_val_acc:.2f}%")
print(f"测试集准确率: {test_acc:.2f}%")

wandb.log({
    "best_val/accuracy": best_val_acc,
    "test/accuracy": test_acc,
})

# ---------- 6. 保存模型 Artifact ----------
# 改动：保存的是验证集上表现最好的那份权重，而不是训练最后一轮的权重
torch.save(best_state, "fashion_mnist_cnn.pth")
artifact = wandb.Artifact(
    name="fashion-mnist-cnn",
    type="model"
)
artifact.add_file("fashion_mnist_cnn.pth")
wandb.log_artifact(artifact)
wandb.finish()


Epoch [1/35], Loss: 1.0833, Val Acc: 86.38%, Best Val: 86.38%, LR: 0.000998
Epoch [2/35], Loss: 0.9340, Val Acc: 87.98%, Best Val: 87.98%, LR: 0.000992
Epoch [3/35], Loss: 0.8898, Val Acc: 89.80%, Best Val: 89.80%, LR: 0.000982
Epoch [4/35], Loss: 0.8595, Val Acc: 89.75%, Best Val: 89.80%, LR: 0.000968
Epoch [5/35], Loss: 0.8407, Val Acc: 90.43%, Best Val: 90.43%, LR: 0.000950
Epoch [6/35], Loss: 0.8289, Val Acc: 91.03%, Best Val: 91.03%, LR: 0.000929
Epoch [7/35], Loss: 0.8174, Val Acc: 91.40%, Best Val: 91.40%, LR: 0.000905
Epoch [8/35], Loss: 0.8085, Val Acc: 91.55%, Best Val: 91.55%, LR: 0.000877
Epoch [9/35], Loss: 0.8002, Val Acc: 92.15%, Best Val: 92.15%, LR: 0.000846
Epoch [10/35], Loss: 0.7911, Val Acc: 91.80%, Best Val: 92.15%, LR: 0.000812
Epoch [11/35], Loss: 0.7896, Val Acc: 92.05%, Best Val: 92.15%, LR: 0.000775
Epoch [12/35], Loss: 0.7810, Val Acc: 92.33%, Best Val: 92.33%, LR: 0.000737
Epoch [13/35], Loss: 0.7773, Val Acc: 92.23%, Best Val: 92.33%, LR: 0.000697
Epoch [1

wandb: uploading artifact fashion-mnist-cnn; updating run metadata
wandb: uploading artifact fashion-mnist-cnn; uploading history steps 34-35, summary
wandb: uploading artifact fashion-mnist-cnn
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: uploading data
wandb: 
wandb: Run history:
wandb: best_val/accuracy ▁
wandb:             epoch ▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
wandb:     learning_rate ██████▇▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁
wandb:     test/accuracy ▁
wandb:        train/loss █▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:      val/accuracy ▁▂▄▄▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇█▇▇▇▇██████████
wandb: 
wandb: Run summary:
wandb: best_val/accuracy 94.03333
wandb:             epoch 35
wandb:     learning_rate 0
wandb:     test/accuracy 93.28
wandb:        train/loss 0.71295
wandb:      val/accuracy 94.03333
wandb: 
wandb:  View run deep-aug-cosine at: https://wandb.ai/3531427435-none/fashion-mnist-cnn/runs/1x3660el
wandb:  View project at: https://wandb.ai/3531427435-none/fashion-mnist-c